# Value function of a given policy — 3x3 Gridworld

Policy:

<p style="text-align: center;">
<img src="./images/01_trajectory_infinite_01.png" width="300" alt="FrozenLake" />
</p>

States are numbered row-major, $s_1,\dots,s_9$:

$$\begin{array}{|c|c|c|}\hline s_1 & s_2 & s_3 \\ \hline s_4 & s_5 & s_6 \\ \hline s_7 & s_8 & s_9 \\ \hline\end{array}$$

* Forbidden (orange) cells: $s_6, s_7$
* Target (blue) cell: $s_9$
* Continuing (infinite-horizon) task, discount factor $\gamma = 0.9$

In [ ]:
import numpy as np

**Parameters**

In [ ]:
# Parameters
gamma = 0.9

**Policy** $\pi$ (deterministic, read from the green arrows)

| state | cell | action | next state $s'$ |
|---|---|---|---|
| $s_1$ | white  | right | $s_2$ |
| $s_2$ | white  | down  | $s_5$ |
| $s_3$ | white  | left  | $s_2$ |
| $s_4$ | white  | right | $s_5$ |
| $s_5$ | white  | down  | $s_8$ |
| $s_6$ | orange | down  | $s_9$ |
| $s_7$ | orange | right | $s_8$ |
| $s_8$ | white  | right | $s_9$ |
| $s_9$ | blue   | stay  | $s_9$ |

As a $9\times 9$ table $\pi(s' \mid s)$ (row = current state, column = next state):

\begin{array}{c|ccc|ccc|ccc}
 & s_1&s_2&s_3&s_4&s_5&s_6&s_7&s_8&s_9\\
\hline
s_1 &0&1&0&0&0&0&0&0&0\\
s_2 &0&0&0&0&1&0&0&0&0\\
s_3 &0&1&0&0&0&0&0&0&0\\
\hline
s_4 &0&0&0&0&1&0&0&0&0\\
s_5 &0&0&0&0&0&0&0&1&0\\
s_6 &0&0&0&0&0&0&0&0&1\\
\hline
s_7 &0&0&0&0&0&0&0&1&0\\
s_8 &0&0&0&0&0&0&0&0&1\\
s_9 &0&0&0&0&0&0&0&0&1
\end{array}

**Transition Matrix** $P_\pi$

Since both the environment and the policy are deterministic, $P_\pi$ contains a single 1 per row.

$$P_\pi = \begin{bmatrix}
0 & 1 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 1 & 0 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 1 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 1 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 1 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 1
\end{bmatrix}$$

In [ ]:
P_pi = np.array([
#  s1 s2 s3 s4 s5 s6 s7 s8 s9
 [0, 1, 0, 0, 0, 0, 0, 0, 0],  # s1 -> right -> s2
 [0, 0, 0, 0, 1, 0, 0, 0, 0],  # s2 -> down  -> s5
 [0, 1, 0, 0, 0, 0, 0, 0, 0],  # s3 -> left  -> s2
 #
 [0, 0, 0, 0, 1, 0, 0, 0, 0],  # s4 -> right -> s5
 [0, 0, 0, 0, 0, 0, 0, 1, 0],  # s5 -> down  -> s8
 [0, 0, 0, 0, 0, 0, 0, 0, 1],  # s6 -> down  -> s9
 #
 [0, 0, 0, 0, 0, 0, 0, 1, 0],  # s7 -> right -> s8
 [0, 0, 0, 0, 0, 0, 0, 0, 1],  # s8 -> right -> s9
 [0, 0, 0, 0, 0, 0, 0, 0, 1],  # s9 -> stay  -> s9
])


**Checking that the sum of the rows are 1 (Stochastic matrix)**

$$\sum_{j=1}^{9} p(s_j \mid s_i) = 1$$

In [ ]:
print(np.sum(P_pi, axis=1))

**Rewards vector**

$$r_\pi = \begin{bmatrix} r_\pi(s_1) \\ r_\pi(s_2) \\ \vdots \\ r_\pi(s_9) \end{bmatrix},
\qquad r_\pi(s) = \sum_a \pi(a\mid s)\, r(s,a)$$

The reward depends on the cell the agent *enters*: $+1$ for the blue target, $0$ for a white cell
($-1$ for a forbidden cell or a boundary hit, but this policy never does either).
Under $\pi$: $s_6 \to s_9$, $s_8 \to s_9$ and $s_9 \to s_9$ collect $+1$; every other state collects $0$.

In [ ]:
#                s1 s2 s3  s4 s5 s6  s7 s8 s9
r_pi = np.array([0, 0, 0,  0, 0, 1,  0, 1, 1]).reshape(-1, 1)

**Identity matrix**

In [ ]:
I = np.eye(9, 9)

**Solving Bellman equation** (with `np.linalg.solve`)

$$(I - \gamma P_\pi)\, v_\pi = r_\pi$$

In [ ]:
M = I - gamma*P_pi
v = np.linalg.solve(M, r_pi)
# print results as a 3x3 matrix
print('value function:\n', np.array2string(v.reshape(3, 3), precision=2))

**Solving Bellman equation iteratively**

$$v_{k+1} = r_\pi + \gamma P_\pi v_k$$

In [ ]:
# initial value (ansatz)
v_old = np.zeros_like(r_pi, dtype=float)
# loop over k
for k in range(200):
    v_new = r_pi + gamma*(P_pi @ v_old)
    # update k -> (k+1)
    v_old = v_new
# print results as a 3x3 matrix
print('value function:\n', np.array2string(v_new.reshape(3, 3), precision=2))